Prepare Training Dataset

In [ ]:
import pandas as pd

# Load cleaned fires & weather
fires = pd.read_csv("fires_clean.csv", parse_dates=["acq_date"])
weather = pd.read_csv("weather.csv", parse_dates=["time"])
grid = pd.read_csv("grid.csv")

# Ensure date columns
fires["date"] = fires["acq_date"].dt.date
weather["date"] = pd.to_datetime(weather["time"]).dt.date

# --- Step 1. Assign fires to grid cells ---
def point_to_cell(lat, lon):
    row = grid[(grid["min_lat"] <= lat) & (lat < grid["max_lat"]) &
               (grid["min_lon"] <= lon) & (lon < grid["max_lon"])]
    if row.empty:
        return None
    return row.iloc[0]["cell_id"]

fires["cell_id"] = fires.apply(lambda r: point_to_cell(r["latitude"], r["longitude"]), axis=1)
fires = fires.dropna(subset=["cell_id"])

# --- Step 2. Create fire presence table ---
fires["fire_today"] = 1
fire_daily = fires.groupby(["cell_id","date"], as_index=False)["fire_today"].max()

# --- Step 3. Shift weather by 1 day (yesterday’s weather → today’s fire) ---
weather["date"] = pd.to_datetime(weather["date"])
weather_shifted = weather.copy()
weather_shifted["date"] = weather_shifted["date"] + pd.Timedelta(days=1)

# --- Step 4. Merge ---
weather_shifted["date"] = pd.to_datetime(weather_shifted["date"]).dt.date
fire_daily["date"] = pd.to_datetime(fire_daily["date"]).dt.date

df = weather_shifted.merge(fire_daily, on=["cell_id","date"], how="left")
df["fire_today"] = df["fire_today"].fillna(0).astype(int)


# Save training dataset
df.to_csv("train_dataset.csv", index=False)

print("Training dataset saved as train_dataset.csv")
print("Shape:", df.shape)
print(df.head())


Training dataset saved as train_dataset.csv
Shape: (77626, 9)
  cell_id       time  temperature_2m_max  temperature_2m_min  \
0      c1 2024-11-01                31.4                17.4   
1      c1 2024-11-02                30.9                16.1   
2      c1 2024-11-03                30.2                16.8   
3      c1 2024-11-04                29.7                16.6   
4      c1 2024-11-05                28.9                18.7   

   precipitation_sum  windspeed_10m_max  relative_humidity_2m_mean  \
0                0.0               14.2                         56   
1                0.0                9.3                         62   
2                0.0                7.5                         69   
3                0.0                7.5                         74   
4                0.0                8.8                         77   

         date  fire_today  
0  2024-11-02           0  
1  2024-11-03           0  
2  2024-11-04           0  
3  2024-11-05       

Model training

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score

# Load dataset
df = pd.read_csv("train_dataset.csv")

# Features & target
feature_cols = [
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "windspeed_10m_max",
    "relative_humidity_2m_mean"
]
X = df[feature_cols].values
y = df["fire_today"].values

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- Logistic Regression ---
lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])
lr.fit(X_train, y_train)
y_prob_lr = lr.predict_proba(X_test)[:, 1]
print("=== Logistic Regression ===")
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))
print(classification_report(y_test, (y_prob_lr >= 0.5).astype(int)))

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
rf.fit(X_train, y_train)
y_prob_rf = rf.predict_proba(X_test)[:, 1]
print("=== Random Forest ===")
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))
print(classification_report(y_test, (y_prob_rf >= 0.5).astype(int)))


=== Logistic Regression ===
ROC-AUC: 0.7339394505461847
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     15295
           1       0.00      0.00      0.00       231

    accuracy                           0.99     15526
   macro avg       0.49      0.50      0.50     15526
weighted avg       0.97      0.99      0.98     15526



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


=== Random Forest ===
ROC-AUC: 0.7690387742365513
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     15295
           1       0.17      0.01      0.02       231

    accuracy                           0.98     15526
   macro avg       0.58      0.51      0.51     15526
weighted avg       0.97      0.98      0.98     15526



In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve

# Retrain Random Forest with more trees
rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    class_weight="balanced_subsample",
    max_depth=15
)
rf.fit(X_train, y_train)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

# Precision-recall curve
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_rf)

# Pick threshold where recall ~0.5
best_idx = np.argmax(recalls >= 0.5)  # first threshold reaching 50% recall
if best_idx < len(thresholds):
    best_thresh = thresholds[best_idx]
else:
    best_thresh = 0.3

print("Chosen threshold:", best_thresh)

y_pred_rf = (y_prob_rf >= best_thresh).astype(int)

print("=== Random Forest (tuned threshold) ===")
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))
print(classification_report(y_test, y_pred_rf))


Chosen threshold: 0.0
=== Random Forest (tuned threshold) ===
ROC-AUC: 0.8420057767230046
              precision    recall  f1-score   support

           0       0.00      0.00      0.00     15295
           1       0.01      1.00      0.03       231

    accuracy                           0.01     15526
   macro avg       0.01      0.50      0.01     15526
weighted avg       0.00      0.01      0.00     15526



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
